# NASDAQ-100 microstructure: what trees can find in order flow, and what they give up to look

[`06_linear`](06_linear.ipynb) fitted a penalized linear map from the microstructure features
at one minute to the return over the next fifteen, and the shape of its penalty sweep says what
kind of feature set this is: **collinear**, with several spread, imbalance and depth measures
carrying nearly the same information at nearly the same lookbacks.

That frames the question for gradient boosting, and the two properties that decide it pull in
opposite directions.

**In its favour, trees represent interactions a linear model cannot.** A linear model sees an
interaction only if someone multiplied two columns together and named the product. A tree
splits on one feature inside a region already defined by others, so an interaction is something
it discovers rather than something it is handed. Order flow is where that should matter: the
same order imbalance means something different in a wide spread than in a tight one, and no
single coefficient on imbalance expresses both.

**Against it, collinearity is hostile to trees in a way it is not to ridge.** Faced with several
near-identical columns, a tree picks one of them at each split, and which one is close to
arbitrary. A dense penalty's advantage on this kind of data comes precisely from *not* choosing
- from spreading weight over the correlated group and averaging their noise down. A greedy
splitter makes that choice at every split and remakes it independently on every fold.

Three dials control how far the fit goes, and this notebook varies all three:

- **Capacity**, set by `num_leaves`: how many regions one tree may carve the feature space into.
  Seven leaves can express a handful of conditions; 63 can express a partition fine enough to
  describe the training window and nothing beyond it. On a fifteen-minute horizon, where almost
  all of the target is noise, the top of this axis is in the grid as a failure case as much as a
  candidate.
- **The loss function**, which decides what "got wrong" means. `mse` minimizes squared error,
  `mae` absolute error, and `huber` behaves like squared error for small residuals and like
  absolute error past a threshold derived from each fold's own label spread. Intraday returns
  are heavy-tailed, so this axis has a mechanism behind it rather than being a sweep for its own
  sake.
- **When to stop**, set by the number of trees. Unlike a linear fit, a boosted model has a
  meaningful state at every iteration, so each configuration is scored at several points along
  its own training run rather than only at the end.

The third dial changes how the results must be read. **A checkpoint is part of a configuration,
not a detail of how it was fitted.** Scoring the declared configurations at several checkpoints
each produces many more candidate models than configurations, and treating that as one candidate
per configuration while quietly keeping each one's best iteration would be reporting the maximum
of several numbers as though it were one.

**Learning objectives.** By the end of this notebook you will be able to:

- Say what a tree ensemble can represent that a penalized linear model cannot, and what a
  collinear feature set costs a greedy splitter.
- Explain why a boosted model produces one result per checkpoint while a linear model produces
  one result in total, and what that implies for counting candidates.
- Read a learning curve of out-of-sample information coefficient against tree count, and tell
  apart a model still learning from one that has begun fitting the training window.
- Say why the choice of loss function is a statement about the label's tails, and relate that to
  what a rank-based metric rewards.
- Recognise that picking each configuration's best checkpoint after seeing the results is a
  selection decision, and locate where selection is actually made.

**Docker image**: `ml4t`

**Book reference**: Chapter 12, Section 12.2 (GBM libraries) and Section 12.3 (how to tune a
boosted model). Chapter 6, Section 6.7 (Search accounting and run logging) introduces the run
log this notebook writes to.

**Prerequisites**: [`03_financial_features`](03_financial_features.ipynb) and
[`04_model_based_features`](04_model_based_features.ipynb) have written the feature matrices,
[`05_evaluation`](05_evaluation.ipynb) has established the walk-forward folds, and
[`06_linear`](06_linear.ipynb) fitted the linear population this one is compared against.

**What it writes**: one training run per configuration and one complete validation prediction
set per configuration and checkpoint, in `run_log/registry.db` and under `run_log/training/` and
`run_log/predictions/`, grouped under a named population.
[`14_backtest`](14_backtest.ipynb) reads that population and selects on validation backtest
Sharpe. **Selection happens there, not here.**

In [ ]:
"""Fit the declared NASDAQ-100 microstructure gradient boosting population on the folds."""

import plotly.graph_objects as go
import polars as pl

from case_studies.research import (
    declared_labels,
    load_model_configs,
    model_requests,
    open_study,
    plan_models,
    run_model_population,
)
from utils.style import COLORS, show_plotly_with_alt

In [ ]:
LABEL = "fwd_ret_15m"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}
CONFIG_NAMES: list[str] = []
POPULATION_NAME = "nasdaq100_microstructure-gbm-validation-v1"
SUPERSEDES_POPULATION: str = ""

In [ ]:
study = open_study(
    "nasdaq100_microstructure", execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None
)

## 1. Which label, and which models

The label is the one the linear notebook used: `fwd_ret_15m`, the return over the fifteen
minutes after the decision minute. Keeping it fixed is what makes the two populations
comparable - the families differ, the target does not. Each label carries its own training menu
at `config/training/{label}.yaml`, so changing `LABEL` above changes which menu is read.

In [ ]:
declared_labels(study, "gbm")

Each name in the menu resolves to a preset in `case_studies/config/lgb/`. The grid is a product
of two axes:

- **Five capacity profiles.** `default` uses the library's own leaf count; the rest fix it at 7,
  15, 31 and 63.
- **Three objectives**, as described above.

Every configuration runs the same number of boosting iterations at the same learning rate, so
the grid isolates capacity and loss rather than confounding them with training length.

In [ ]:
configs = load_model_configs(
    study,
    "gbm",
    labels=[LABEL],
    config_names=CONFIG_NAMES or None,
)
configs

## 2. Binding the declarations to the data

Binding a declaration to the data means finding the fold boundaries, the exact rows each fit
must predict, and the value of every data-dependent parameter. Huber's threshold is one of
those: it is a fraction of the training labels' standard deviation, so it is a different
number on every fold and comes from that fold's own data.

`plan_models` derives all of it - including every identity below - without holding it. That
distinction is what makes this notebook runnable on a minute panel: a *resolved* request
carries its prepared folds, so resolving the whole declaration up front would hold the same
standardized design matrix once per configuration. Execution instead walks folds on the
outside and configurations on the inside, so one fold set is live at a time.

Nothing is fitted here, so the plan can be read first. Three things to check:

- **One row per configuration and checkpoint.** `checkpoint_value` is where this differs from
  the linear plan: a boosted model publishes predictions from several training states, not
  one, so each configuration contributes as many rows as it has checkpoints. The row count is
  the number of candidate models this notebook is about to create.
- **`training_hash` repeats across the checkpoints of one configuration**, because they come
  from a single fit. It is the fit that is identified, and the checkpoint that selects from it.
- **`prediction_hash` is unique on every row**, since each is a different set of predictions.

In [ ]:
requests = model_requests(
    study,
    configs,
    execution_tier=EXECUTION_TIER,
    preview_reductions=PREVIEW_REDUCTIONS,
)
plan = plan_models(study, requests=requests)

pl.DataFrame(
    {
        "config_name": [member.config_name for member in plan.members],
        "checkpoint_kind": [member.checkpoint_kind for member in plan.members],
        "checkpoint_value": [member.checkpoint_value for member in plan.members],
        "training_hash": [member.training_hash for member in plan.members],
        "prediction_hash": [member.prediction_hash for member in plan.members],
    }
)

## 3. Fitting the population

`run_model_population` fits every resolved request. For one request it walks the folds, and on
each one:

1. takes the rows inside that fold's training window,
2. casts the design matrix to the precision LightGBM works in and leaves missing values in
   place - a tree routes a missing value down its own branch, so imputing a median here would
   hand the model an observation nobody made,
3. fits the declared number of boosting iterations,
4. predicts the fold's validation rows at each checkpoint, using only the trees built up to that
   iteration.

Step 4 is what makes one fit produce many results. The fold predictions are concatenated into
one series per checkpoint covering the whole validation period, and each becomes its own
registered prediction set with its own identity.

Preparation happens once per fold and is shared by every configuration, because slicing the
window and cleaning the rows depends on the data and not on the model. The run walks folds on
the outside and configurations on the inside for the same reason: one prepared fold is held at a
time rather than the whole set. On this case study that is not a nicety - the panel is minute
bars across the constituents, and holding every prepared fold at once is the difference between
a run that fits in memory and one that does not.

**What the call publishes is a population**: a named, immutable list of the prediction sets it
will produce, written down before the first fit. Afterwards every member must exist and be
complete, which is what makes the downstream comparison well defined.

`SUPERSEDES_POPULATION` names the population hash this run replaces. A changed estimator
parameter moves every training identity as surely as a changed menu does, so a refit is a
different population under the same name, and `OfficialPopulation.create` refuses to write it
without being told which snapshot it retires. That declaration is the only record of which
generation is which, and a run that omits it fails at the publish - after every fit has been
paid for.

In [ ]:
execution, population = run_model_population(
    study, plan, population_name=POPULATION_NAME, supersedes=SUPERSEDES_POPULATION or None
)

print(f"{len(execution.runs)} configurations fitted")
print(f"population {population.name}: {len(population.members)} prediction sets")

Re-running this notebook unchanged costs the time it takes to read the data. Every identity is
re-derived from the inputs, the registry already holds the matching rows, and the runner returns
the stored result rather than fitting again.

### Running configurations of your own

The published run log is read-only. To add runs, open the study against a workspace, which holds
its own registry and artifacts and reads the same labels and features:

```python
study = open_study("nasdaq100_microstructure", workspace="~/ml4t-experiments")
configs = load_model_configs(
    study, "gbm", labels=["fwd_ret_15m"], config_names=["leaves_15_huber", "leaves_31_huber"]
)
requests = model_requests(study, configs)
plan = plan_models(study, requests=requests)
execution, population = run_model_population(study, plan, population_name="my-gbm-v1")
```

`CONFIG_NAMES` fits a subset of what the menu declares. To fit something new, add a preset at
`case_studies/config/lgb/leaves_127_huber.yaml` and list `leaves_127_huber` under `gbm:` in the
label's menu. Editing an existing preset changes that configuration's identity, so its result
registers as a new row beside the old one rather than replacing it.
[`RUN_LOG.md`](../RUN_LOG.md#running-your-own-configurations) covers the rest.

## 4. What came out

One row per configuration and checkpoint. `ic_mean` is the **information coefficient**: at each
decision time, rank the constituents by the model's prediction, rank them by the return they
went on to earn over the next fifteen minutes, correlate the two rankings, and average that
correlation over the validation period.

The table is sorted by IC, and the top of it is the trap this notebook exists to describe. The
leading row is the maximum over every configuration *and* every checkpoint. Reading it as the
result of one experiment would attribute to the model whatever the stopping point contributed,
and the section below measures how large that contribution is before anything is concluded from
the ranking.

`ic_n_days` carries the second warning, and it is the one `06_linear` turned on: a configuration
that scored fewer decision times than its neighbours is not comparable to them, because its IC
is an average over the times where it stayed non-degenerate. Every comparison below is
restricted to full-coverage members for that reason. As in the linear notebook, the column
counts decision times rather than days: this case study decides every fifteen minutes.

In [ ]:
catalog = execution.catalog_rows.select(
    "config_name",
    "label",
    "complete",
    "checkpoint_value",
    "ic_mean",
    "ic_std",
    "ic_n_days",
    "n_folds",
    "training_hash",
    "prediction_hash",
).sort("ic_mean", descending=True)

if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("gbm execution returned a partial prediction set")

full_times = int(catalog.get_column("ic_n_days").max())
catalog = catalog.with_columns(full_coverage=pl.col("ic_n_days") == full_times)

print(f"{catalog.height} candidate models: {catalog.n_unique('config_name')} configurations")
print(f"at {catalog.n_unique('checkpoint_value')} checkpoints each")
print(f"full coverage is {full_times:,} scored decision times")
catalog.select(
    "config_name",
    "checkpoint_value",
    "ic_mean",
    "ic_std",
    "ic_n_days",
    "full_coverage",
).head(15)

### What more trees do

Each line traces one configuration's out-of-sample IC as trees are added to it. This is the
figure the checkpoint dimension exists to produce, and it separates two things a single
end-of-training number cannot.

A line that rises and then falls has an interior optimum: the model was still learning, then
began fitting the training window at the expense of the validation folds. A line that wanders
without trend around zero never had anything to learn in the first place, and its highest point
is wherever the noise happened to peak. The difference matters, because both produce a
respectable-looking maximum.

In [ ]:
curves = catalog.filter("full_coverage").sort("config_name", "checkpoint_value")
objectives = {"mse": COLORS["blue"], "mae": COLORS["amber"], "huber": COLORS["copper"]}


def objective_of(name: str) -> str:
    """Read the loss function out of a declared configuration name."""
    return next((key for key in objectives if name.endswith(key)), "mse")


fig_curves = go.Figure()
for config_name in curves.get_column("config_name").unique(maintain_order=True):
    series = curves.filter(pl.col("config_name") == config_name)
    fig_curves.add_trace(
        go.Scatter(
            x=series.get_column("checkpoint_value").to_list(),
            y=series.get_column("ic_mean").to_list(),
            mode="lines",
            name=config_name,
            line=dict(color=objectives[objective_of(config_name)], width=1.5),
            opacity=0.75,
        )
    )
fig_curves.add_hline(y=0, line_width=1, line_dash="dash", line_color=COLORS["neutral"])
fig_curves.update_layout(
    title="Validation IC against boosting iteration, by loss function",
    height=550,
    width=1000,
    margin=dict(t=70),
    legend=dict(font=dict(size=9)),
)
fig_curves.update_xaxes(title_text="Boosting iterations (trees kept)")
fig_curves.update_yaxes(title_text="Mean cross-sectional IC (validation)")
show_plotly_with_alt(
    fig_curves,
    "Line chart of mean validation information coefficient against boosting iteration, one line "
    "per configuration, coloured by loss function: blue for squared error, amber for absolute "
    "error, copper for Huber. A dashed line marks zero.",
)

### Whether the loss function is what separates them

The curves are coloured by objective because that is the axis with a mechanism behind it.
Fifteen-minute returns are heavy-tailed, so if the extremes are steering the squared-error fits,
the three colours should separate, and they should separate more as trees are added, since each
additional tree is fitted to the residuals the previous ones left.

The chart below drops the checkpoint dimension by taking each configuration's final state, so
every configuration is compared at the same amount of training. That is the comparison that does
not require choosing anything after the fact.

In [ ]:
final_iteration = int(catalog.get_column("checkpoint_value").max())
final = (
    catalog.filter(pl.col("checkpoint_value") == final_iteration)
    .filter("full_coverage")
    .with_columns(objective=pl.col("config_name").map_elements(objective_of, return_dtype=pl.Utf8))
    .sort("ic_mean", descending=True)
)

fig_obj = go.Figure(
    go.Bar(
        x=final.get_column("config_name").to_list(),
        y=final.get_column("ic_mean").to_list(),
        marker_color=[objectives[value] for value in final.get_column("objective")],
        text=[f"{value:+.4f}" for value in final.get_column("ic_mean")],
        textposition="outside",
        cliponaxis=False,
    )
)
fig_obj.add_hline(y=0, line_width=1, line_dash="dash", line_color=COLORS["neutral"])
fig_obj.update_layout(
    title="Validation IC at the final iteration, coloured by loss function",
    height=500,
    width=1000,
    showlegend=False,
    margin=dict(t=70),
)
fig_obj.update_xaxes(title_text="Configuration (sorted by validation IC)", tickangle=-45)
fig_obj.update_yaxes(title_text="Mean cross-sectional IC (validation)")
show_plotly_with_alt(
    fig_obj,
    "Bar chart of mean validation information coefficient for every full-coverage configuration "
    "at its final boosting iteration, sorted descending and coloured by loss function, against a "
    "dashed zero line.",
)

### How much the checkpoint moves a configuration

One number per configuration: the range its IC covers across its own checkpoints. This is the
quantity that decides whether choosing a stopping point is a decision worth making carefully or
one being made by noise. A configuration whose IC varies more across its own training run than
the configurations vary among themselves is one where the checkpoint, not the model, is doing
the ranking.

In [ ]:
spread = (
    curves.group_by("config_name")
    .agg(
        ic_min=pl.col("ic_mean").min(),
        ic_max=pl.col("ic_mean").max(),
        ic_final=pl.col("ic_mean").filter(pl.col("checkpoint_value") == final_iteration).first(),
    )
    .with_columns(checkpoint_range=pl.col("ic_max") - pl.col("ic_min"))
    .sort("checkpoint_range", descending=True)
)
across_configs = float(final.get_column("ic_mean").max() - final.get_column("ic_mean").min())
within_config = float(spread.get_column("checkpoint_range").median())
print(f"IC range across configurations at the final iteration: {across_configs:.4f}")
print(f"median IC range within one configuration: {within_config:.4f}")
print(f"the checkpoint moves a configuration {within_config / across_configs:.2f}x as far")
spread

## 5. What to notice

**Read the two comparisons in the right order.** The fixed-iteration bar chart compares fifteen
models at the same amount of training, with nothing chosen after the fact, and it is the honest
ranking. The learning curves say whether that ranking is being taken at a sensible point: if
every configuration has already peaked well before the final iteration, the declared training
length is longer than this data supports, and the bar chart is comparing fifteen models in their
overfitted regime. That comparison is still fair - the same amount of training for everyone -
but the ranking it produces is not the ranking their best states would produce.

**The checkpoint range against the configuration range is the number that matters.** It is
printed above. If a single configuration's IC varies across its own training run by a comparable
amount to how much the fifteen configurations vary among themselves at fixed training length,
then a stopping point chosen after seeing the curves would be doing as much work as the choice
of model. That is why reporting the leading row of the results table would be reporting a
maximum over configurations and checkpoints together as though it were one experiment.

**The loss function is a claim about which errors matter.** Squared error weights an observation
by the square of its error, so the largest fifteen-minute moves dominate what each successive
tree is fitted to, while the information coefficient is a rank correlation that cares about
order rather than magnitude: effort spent getting the extremes right buys nothing on this
metric. Whether the three colours separate in the charts above is the test of whether that
mechanism is active on this data, and it is worth matching the objective to the metric the
result will be judged on.

**Compare this population against the linear one on the same terms.** Both were fitted on the
same label, the same features and the same folds, so the strongest full-coverage member here and
the strongest in [`06_linear`](06_linear.ipynb) are directly comparable. If boosting does not
win, the collinearity argument at the top of this notebook is the first thing to weigh rather
than a tuning failure: a dense penalty earns its result by refusing to choose among
near-duplicate columns, and a greedy splitter chooses at every split.

**None of this selects anything.** IC measures whether predictions rank names correctly, not
whether a strategy trading them makes money after costs and turnover. At a fifteen-minute
horizon that gap is at its widest, because the position turns over constantly and costs decide
the outcome. Selection is on validation backtest Sharpe over the population just published, and
it happens in [`14_backtest`](14_backtest.ipynb), where the checkpoint is part of what is
selected.

**Known limitations.** The IC is an average of per-timestamp rank correlations with no
adjustment for the serial dependence overlapping fifteen-minute returns create, so it is a
diagnostic rather than a test, and it carries no interval that would say whether these
configurations differ from each other or from the linear ones. The grid varies capacity and loss
at a fixed learning rate and a fixed training length, so it says nothing about trading one
against another. And every number is measured on validation folds that have been read many times
over by the time a case study reaches this notebook.

**Next**: [`08_dl_nlinear`](08_dl_nlinear.ipynb) starts the sequence of neural architectures,
which represent the same order flow a third way again. The useful thing to watch is whether any
of them recovers structure the trees found here, and whether a collinear feature set costs them
what it costs a greedy splitter.